# Baseline 2: logistinė regresija

**Uždavinys:** prognozuoti $\hat p(y=1\mid x)$ — tikimybę, kad klientas sutiks su terminuotu indėliu.

Šis notebook yra **savarankiškas**: Colab užtenka įkelti `bank-full.csv` šalia notebook'o. Kiti 3 modeliai yra kituose failuose.

Fiksuota sėkla `SEED=42` visur.

## 0. Bibliotekos

Čia įkeliame pandas/sklearn. Logistinei regresijai CatBoost nereikia.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve

from sklearn.linear_model import LogisticRegression
from IPython.display import display


## 1. Bendros taisyklės (sėkla, keliai, metrikos)

Kodėl fiksuojame sėklą: dokumente reikalaujama, kad **ta pati komanda du kartus** duotų tuos pačius skaičius.

Kodėl `bank-full.csv` be Windows kelio: Google Colab nemato `C:\Users\...` — failas turi būti darbo aplanke.

In [ ]:
# Fiksuota sėkla visur — paleidus du kartus rezultatai turi sutapti.
SEED = 42
C_CALL = 1.0      # sąlyginis vieno skambučio kaštas
V_SUCCESS = 10.0  # sąlyginė sėkmingo indėlio vertė (parametrai derinami su banku)
K_FRACTION = 0.10 # precision@k: k = 10 % test imties (ribotas operatorių biudžetas)

import os, random
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "bank-full.csv"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAT_COLS = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
NUM_BASE = ["age","balance","day","campaign","pdays","previous","never_contacted"]

def load_bank(path=None):
    path = path or DATA_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Nerastas {path}. Colab: kairėje Files (aplankas) → Upload → įkelkite bank-full.csv "
            "į tą pačią sesiją kaip šis notebook."
        )
    df = pd.read_csv(path, sep=";")
    print(f"Nuskaityta: {df.shape[0]} eil. × {df.shape[1]} stulp.")
    return df

def month_change_indices(df):
    m = df["month"].to_numpy()
    ch = [0]
    for i in range(1, len(m)):
        if m[i] != m[i-1]:
            ch.append(i)
    ch.append(len(df))
    return ch

def chronological_split(df):
    """~70/15/15 pagal eilutės tvarką, ribos prie mėnesio virsmo."""
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train, val, test = df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()
    print(f"Skaidymas train/val/test: {len(train)}/{len(val)}/{len(test)}  ribos={train_end},{val_end}")
    print(f"  train month nuo {train['month'].iloc[0]} iki {train['month'].iloc[-1]}")
    print(f"  val   month nuo {val['month'].iloc[0]} iki {val['month'].iloc[-1]}")
    print(f"  test  month nuo {test['month'].iloc[0]} iki {test['month'].iloc[-1]}")
    return train, val, test

def temporal_shift_split(df):
    """
    Tas pats mokymas (pirmos ~70 %). Du testai:
    arti = vidurys (~15 %), toli = pabaiga (~15 %).
    """
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train = df.iloc[:train_end].copy()
    near = df.iloc[train_end:val_end].copy()
    far = df.iloc[val_end:].copy()
    print(
        f"Laiko poslinkis: mokymas n={len(train)}; "
        f"arti (vidurys) n={len(near)}; toli (pabaiga) n={len(far)}"
    )
    return train, near, far

def add_features(df):
    out = df.copy()
    # pdays=-1: klientas niekada nekontaktuotas anksčiau → atskiras požymis,
    # o -1 pakeičiame 0, kad tai nebūtų „neigiama trukmė“.
    out["never_contacted"] = (out["pdays"] == -1).astype(int)
    out.loc[out["pdays"] == -1, "pdays"] = 0
    out["y_bin"] = (out["y"].astype(str).str.lower() == "yes").astype(int)
    return out

def cap_previous(train, *others):
    cap = float(train["previous"].quantile(0.99))
    print(f"previous apkirpimas ties train 99-uoju procentiliu = {cap:.2f} (max buvo {train['previous'].max()})")
    out = []
    for p in (train,) + others:
        q = p.copy()
        q["previous"] = q["previous"].clip(upper=cap)
        out.append(q)
    return tuple(out)

def one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(include_duration):
    num = NUM_BASE + (["duration"] if include_duration else [])
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", one_hot(), CAT_COLS),
    ], remainder="drop")

def Xy(df, include_duration):
    cols = CAT_COLS + NUM_BASE + (["duration"] if include_duration else [])
    return df[cols].copy(), df["y_bin"].to_numpy()

def precision_at_k(y, p, k):
    k = int(min(k, len(y)))
    order = np.argsort(-np.asarray(p), kind="mergesort")
    return float(np.asarray(y)[order][:k].mean())

def contact_cost(y, yhat, c_call=C_CALL, v_success=V_SUCCESS):
    y = np.asarray(y).astype(int); yhat = np.asarray(yhat).astype(int)
    tp = int(((yhat==1)&(y==1)).sum()); fp = int(((yhat==1)&(y==0)).sum())
    return float(c_call*(tp+fp) - v_success*tp)

def best_threshold(y_val, p_val):
    cands = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 19)))
    best_t, best_c = 0.5, np.inf
    for t in cands:
        c = contact_cost(y_val, (p_val >= t).astype(int))
        if c < best_c:
            best_c, best_t = c, float(t)
    return best_t

def metrics_dict(y, p, thr, k=None):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    if k is None:
        k = max(1, int(K_FRACTION * len(y)))
    yhat = (p >= thr).astype(int)
    return {
        "PR-AUC": float(average_precision_score(y, p)),
        "precision@k": precision_at_k(y, p, k),
        "k": int(k),
        "Brier": float(brier_score_loss(y, p)),
        "kaštai": contact_cost(y, yhat),
        "slenkstis": float(thr),
        "n": int(len(y)),
        "positives": float(y.mean()),
    }

def plot_pr(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        pr, rc, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        plt.plot(rc, pr, label=f"{name} (PR-AUC={ap:.3f})")
    plt.xlabel("Atgaminimas (Recall)"); plt.ylabel("Tikslumas (Precision)")
    plt.title(title); plt.legend(loc="lower left"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def plot_cal(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        frac, mp = calibration_curve(y, p, n_bins=10, strategy="quantile")
        plt.plot(mp, frac, marker="o", label=f"{name} (Brier={brier_score_loss(y,p):.3f})")
    plt.plot([0,1],[0,1],"k--", label="ideali kalibracija")
    plt.xlabel("Vidutinė p̂"); plt.ylabel("Stebėta teigiamų dalis")
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def save_preds(name, y, p):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, f"preds_{name}.csv")
    pd.DataFrame({"y": y, "p": p}).to_csv(path, index=False)
    print("Išsaugota", path)

def load_other_preds(exclude):
    found = {}
    if not os.path.isdir(OUTPUT_DIR):
        return found
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if fn.startswith("preds_") and fn.endswith(".csv"):
            key = fn[len("preds_"):-4]
            if key == exclude:
                continue
            tab = pd.read_csv(os.path.join(OUTPUT_DIR, fn))
            if "y" in tab.columns and "p" in tab.columns:
                found[key] = (tab["y"].to_numpy(), tab["p"].to_numpy())
    return found

def show_probability_examples(te_df, y_true, p_hat, thr, n=10, model_name="model"):
    """
    Parodo, kaip p̂ naudojama praktiškai: operatorius skambina nuo didžiausios
    tikimybės. 10 test klientų + histograma visai test imčiai.
    „Teisus/klydo“ lyginama su kaštais parinktu slenksčiu thr (ne su 0,5),
    nes prie 11,7 % teigiamos klasės p̂ dažnai būna mažesnė nei 0,5.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tab = te_df.copy()
    tab["p_hat"] = np.asarray(p_hat, dtype=float)
    tab["y_tikras"] = np.asarray(y_true).astype(int)
    tab["eilutes_nr"] = tab.index.astype(int)
    top = tab.sort_values("p_hat", ascending=False).head(n)
    cols = [
        c for c in [
            "eilutes_nr", "age", "job", "marital", "education", "balance",
            "housing", "loan", "contact", "month", "campaign", "poutcome",
            "never_contacted", "p_hat", "y_tikras",
        ]
        if c in top.columns
    ]
    print("10 test klientų, surikiuotų mažėjančia p̂ (kaip skambučių sąrašas):")
    display(top[cols].reset_index(drop=True))

    print("\nTas pats paprastais sakiniais:")
    for _, row in top.iterrows():
        p = float(row["p_hat"])
        tikras = "taip" if int(row["y_tikras"]) == 1 else "ne"
        pred_taip = p >= thr
        actual_taip = int(row["y_tikras"]) == 1
        verdiktas = "modelis teisus" if pred_taip == actual_taip else "modelis klydo"
        print(
            f"Klientas Nr. {int(row['eilutes_nr'])}: p̂={p:.2f} → {100*p:.0f}% "
            f"tikimybė sutikti, tikras atsakymas: {tikras} ({verdiktas})."
        )

    plt.figure(figsize=(7, 4))
    plt.hist(np.asarray(p_hat), bins=20, range=(0, 1), edgecolor="black", color="steelblue")
    plt.xlabel("p̂ (tikimybė, kad sutiks)")
    plt.ylabel("Kiek klientų")
    plt.title("Kaip pasiskirsto visos test imties tikimybės")
    plt.xlim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"prob_hist_{model_name}.png")
    plt.savefig(fname, dpi=140)
    plt.show()
    print("Histograma išsaugota:", fname)


def prepare_parts(include_duration=False):
    raw = load_bank()
    print("y=yes dalis visame rinkinyje:", (raw["y"].str.lower()=="yes").mean())
    print("pdays=-1 dalis:", (raw["pdays"]==-1).mean(), "(dokumente ~81,7 %)")
    print("duration=0 eilučių:", (raw["duration"]==0).sum(), "(visos turėtų būti y=no — nutekėjimo požymis)")
    ch = month_change_indices(raw)
    print("Mėnesio periodų:", len(ch)-1)
    df = add_features(raw)
    tr, va, te = chronological_split(df)
    tr, va, te = cap_previous(tr, va, te)
    print(
        "y=1 dalis train/val/test:",
        round(tr["y_bin"].mean(), 4),
        round(va["y_bin"].mean(), 4),
        round(te["y_bin"].mean(), 4),
        "← vėlesnėse kampanijose sutarčių daugiau; todėl chronologija būtina.",
    )
    return tr, va, te


## 2. Duomenų gavimas ir chronologijos patikra

`bank-full.csv` eilutės jau surikiuotos pagal datą (2008 gegužė – 2010 lapkritis). Todėl **nemaišome** eilučių atsitiktinai — kitaip testas „matytų ateitį“.

In [ ]:
tr, va, te = prepare_parts(include_duration=False)
print(tr.head(3))


## 3. Požymių paruošimas

- **one-hot:** kiekviena kategorija (`job=student`, `poutcome=success`, `unknown` ir t. t.) tampa 0/1 stulpeliu. `unknown` paliekame kaip tikrą kategoriją, nes tai informacija („nežinome kontakto tipo“).
- **never_contacted:** `pdays=-1` (apie 81,7 % eilučių) nėra trukmė, o vėliavėlė „niekada neskambinta anksčiau“.
- **previous apkirpimas:** maks. 275, vidurkis ~0,58 — vienas išskirtis iškreiptų svorius; kerpame pagal **train** 99-ąjį procentilį, kad val/test statistika nenutekėtų į mokymą.
- **duration išjungtas:** trukmė paaiškėja tik **po** pokalbio, todėl realistiniame modelyje jos nėra.

In [ ]:
include_duration = False
prep = make_preprocessor(include_duration)
X_tr = prep.fit_transform(Xy(tr, include_duration)[0])
X_va = prep.transform(Xy(va, include_duration)[0])
X_te = prep.transform(Xy(te, include_duration)[0])
y_tr, y_va, y_te = tr["y_bin"].to_numpy(), va["y_bin"].to_numpy(), te["y_bin"].to_numpy()
print("Požymių po one-hot:", X_tr.shape[1])
print("Train teigiamų dalis:", y_tr.mean())


## 4. Modelio mokymas

Logistinė regresija skaičiuoja **vieną tiesinį balą** ir paverčia jį tikimybe sigmoide: $\hat p=\sigma(w^\top x+b)$.

`class_weight='balanced'` padidina retos klasės (sutiko, 11,7 %) svorį, kad modelis jos „nepamirštų“. Tai ne persamplingas, o svoriai nuostolio funkcijoje.

In [ ]:
MODEL_NAME = "baseline"
model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=SEED,
)
model.fit(X_tr, y_tr)
p_va = model.predict_proba(X_va)[:, 1]
p_te = model.predict_proba(X_te)[:, 1]
thr = best_threshold(y_va, p_va)
print("Slenkstis pagal min. kaštus ant validation:", round(thr, 4))


## 4.1 Tikimybės paprastai

`predict_proba` grąžina skaičių nuo 0 iki 1 kiekvienam klientui. Čia paimame 10 test klientų, **surikiuojame nuo didžiausios p̂** (taip operatorius ir gautų skambučių sąrašą) ir palyginame su tikru atsakymu. Histograma parodo, ar modelis visiems duoda panašią tikimybę, ar moka atskirti „beveik tikrai sutiks“ nuo „beveik tikrai ne“.

In [ ]:
show_probability_examples(te, y_te, p_te, thr, n=10, model_name=MODEL_NAME)


## 5. Vertinimas teste

- **PR-AUC** tinka disbalansui geriau nei accuracy (88 % „no“ duotų aukštą tikslumą nieko neskambinant).
- **precision@k** — koks pataikymas, jei operatorius spėja paskambinti tik 10 % geriausių pagal $\hat p$.
- **Brier + kalibracijos kreivė** — ar $\hat p=0.7$ tikrai reiškia ~70 % sutarčių.
- **Kaštai** $= c_{call}\cdot(TP+FP)-v_{success}\cdot TP$.

In [ ]:
m = metrics_dict(y_te, p_te, thr)
display(pd.DataFrame([m], index=[MODEL_NAME]))
save_preds(MODEL_NAME, y_te, p_te)
plot_pr({MODEL_NAME: (y_te, p_te)}, "PR kreivė — logistinė regresija", "pr_logreg.png")
plot_cal({MODEL_NAME: (y_te, p_te)}, "Kalibracija — logistinė regresija", "calibration_logreg.png")


## 6. Abliacija ir laiko poslinkis

Abliacija parodo, kiek „stebuklingai“ padeda `duration` (nutekėjimas).

Laiko poslinkis: **tas pats** modelis (mokytas ant pirmų ~70 %). Lyginame vidurį (netolima ateitis) su pabaiga (vėlyvos kampanijos). Pagrindinis testas jau ir yra pabaiga — čia svarbu, ar viduryje sekasi geriau nei gale.

In [ ]:
print("="*60)
print("ABLIACIJA: tas pats modelis SU duration ir BE duration")
print("duration žinomas tik po skambučio, todėl realistinis modelis jo nenaudoja.")
print("="*60)

# --- be duration (jau turime p_te) ---
# --- su duration ---
prep_d = make_preprocessor(True)
Xtr_d = prep_d.fit_transform(Xy(tr, True)[0])
Xva_d = prep_d.transform(Xy(va, True)[0])
Xte_d = prep_d.transform(Xy(te, True)[0])
ytr_d, yva_d, yte_d = tr["y_bin"].to_numpy(), va["y_bin"].to_numpy(), te["y_bin"].to_numpy()

model_d = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="lbfgs", random_state=SEED)
model_d.fit(Xtr_d, ytr_d)

p_te_d = model_d.predict_proba(Xte_d)[:, 1]
ap_no = average_precision_score(y_te, p_te)
ap_d = average_precision_score(yte_d, p_te_d)
print(f"PR-AUC be duration = {ap_no:.4f}")
print(f"PR-AUC su duration = {ap_d:.4f}")
print(f"Skirtumas (su − be) = {ap_d-ap_no:.4f}  ← didelis šuolis = duomenų nutekėjimas")
plot_pr(
    {"be duration": (y_te, p_te), "su duration": (yte_d, p_te_d)},
    "Abliacija: duration poveikis PR kreivei",
    f"pr_ablation_{MODEL_NAME}.png",
)

print("="*60)
print("ATSPARUMO BANDYMAS: laiko poslinkis")
print("Tas pats modelis išmokytas ant pirmų ~70 % eilučių.")
print("Arti = vidurys (netolima ateitis). Toli = pabaiga (vėlyvos kampanijos).")
print("Abiejoms pusėms slenkstis 0,5 — kad kaštai būtų palyginami. PR-AUC nuo slenksčio nepriklauso.")
print("="*60)
print(f"y=1 dažnis arti = {y_va.mean():.4f}  ({int(y_va.sum())}/{len(y_va)})")
print(f"y=1 dažnis toli = {y_te.mean():.4f}  ({int(y_te.sum())}/{len(y_te)})")
m_near = metrics_dict(y_va, p_va, 0.5)
m_far = metrics_dict(y_te, p_te, 0.5)
cmp = pd.DataFrame([m_near, m_far], index=["arti laike (vidurys)", "toli laike (pabaiga)"])
display(cmp[["n", "positives", "PR-AUC", "precision@k", "Brier", "kaštai"]])
print(f"PR-AUC (arti − toli) = {m_near['PR-AUC'] - m_far['PR-AUC']:.4f}")
print("Jei toli PR-AUC aiškiai prastesnis — taisyklės iš ankstyvų kampanijų vėliau nebegalioja (concept drift).")
plot_pr(
    {"arti (vidurys)": (y_va, p_va), "toli (pabaiga)": (y_te, p_te)},
    "Laiko poslinkis: tas pats modelis, dvi ateitys",
    f"pr_timeshift_{MODEL_NAME}.png",
)


## 7. Palyginimas su kitais modeliais

Jei tame pačiame aplanke jau paleisti CatBoost / SVM / MLP notebook'ai, čia susirinks bendra lentelė ir grafikai.

In [ ]:
# Palyginimas su kitais modeliais, jei jų notebook'ai jau paleisti (outputs/preds_*.csv)
others = load_other_preds(MODEL_NAME)
all_curves = {MODEL_NAME: (y_te, p_te), **others}
if len(all_curves) > 1:
    print("Rasti kiti modeliai:", list(others.keys()))
    rows = []
    for name,(yy,pp) in all_curves.items():
        # slenkstis 0.5 palyginimui tarp failų; tikrosios metrikos — kiekvieno notebook viduje
        rows.append({"modelis": name, **metrics_dict(yy, pp, 0.5)})
    display(pd.DataFrame(rows).set_index("modelis"))
    plot_pr(all_curves, "Precision–Recall: visi rasti modeliai", "pr_all_models.png")
    plot_cal(all_curves, "Kalibracija: visi rasti modeliai", "calibration_all_models.png")
else:
    print("Kitų modelių preds_*.csv nėra. Paleiskite kitus 3 notebook'us tame pačiame aplanke, tada perleiskite šią celę — atsiras bendri grafikai.")


Paleidus šią celę, į jūsų kompiuterio Atsisiuntimų (Downloads) aplanką atsisiųs ZIP failas su visais šio modelio rezultatais (preds_*.csv ir grafikais).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"{MODEL_NAME}_outputs", "zip", "outputs")
files.download(f"{MODEL_NAME}_outputs.zip")
